# Cosmos → Leader Record linkage — V18 (rule-first selection)

**Implementation based on V17 full-population notebook.** V18 keeps the same source tables, employer RCID gate, title normalization, role-level checks, executive-office/COO safeguards, date eligibility and candidate status definitions. **Only rule assignment, ranking, and related diagnostics are revised.**

## Confirmed V18 ranking

`RULE_NO ASC → MATCH_TIER ASC → TITLE_SIM DESC → COUNTRY_SORT ASC → DAYS_FROM_POSTING ASC → PERSON_ID ASC`

- `RULE_NO` is the **primary** criterion (R1 highest, R8 lowest).
- `MATCH_TIER`: 1 = exact / similarity ≥98; 2 = similarity 95–97; 3 = qualifying word match.
- `COUNTRY_SORT`: `YES` preferred; `NO` and `UNKNOWN` equal.
- `DAYS_FROM_POSTING`: absolute days from **POSTING_MONTH** to role start, as in V17.
- `PERSON_ID` only breaks unresolved ties for display; tied status is based on criteria 1–5.

## Eight rules (evaluated in this exact order)

| Rule | Title condition | Timing condition |
|---|---|---|
| R1 | Tier 1 | 0–6 months after posting month |
| R2 | Tier 1 | January 1 of posting year |
| R3 | Tier 2 or 3 | 0–6 months after |
| R4 | Tier 2 or 3 | January 1 fallback |
| R5 | Tier 1 | 0–12 months after |
| R6 | Tier 2 or 3 | 0–12 months after |
| R7 | Tier 1 | 0–6 months before posting month |
| R8 | Tier 2 or 3 | 0–6 months before |

**Rule exclusivity:** first matching rule wins. R5/R6 effectively capture eligible month 6–12 starts not already captured by R1/R3. **Strong** in R3/R4/R6/R8 explicitly means *Tier 2 or Tier 3*, not Tier 1. A title must pass the existing V17 safeguards before entering any rule.

**Execution:** Run all cells top-to-bottom in a Snowflake notebook. Output objects are versioned `LNK_V18_*`; V17 objects remain unchanged. No Snowflake execution or V18 accuracy measurement has been performed in this exported notebook.

## 0 · Snowflake session and parameters

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# ============================ CONFIG ============================
WORK_SCHEMA = "PC_FIVETRAN_DB.DA_WORKSPACE"            # all outputs (LNK_* tables and functions) are written here
WAREHOUSE   = None                                      # e.g. "ANALYTICS_WH_L"; None keeps the current one
SRC_COSMOS  = "PC_FIVETRAN_DB.DA_WORKSPACE.V_COSMOS_SENIOR_LEADERSHIP_CONSUMPTION"
SRC_LEADER  = "PC_FIVETRAN_DB.DA_WORKSPACE.LEADER_RECORD"
SIM_BAR     = 95                                        # title similarity threshold for the non-exact rules
NEAR_EXACT_BAR = 98                                     # similarity at or above this ranks like an exact title
# ================================================================

if WAREHOUSE:
    session.sql(f"USE WAREHOUSE {WAREHOUSE}").collect()
session.sql(f"USE SCHEMA {WORK_SCHEMA}").collect()
session.sql(f"SET SRC_COSMOS = '{SRC_COSMOS}'").collect()   # used by the SQL cells via IDENTIFIER($SRC_COSMOS)
session.sql(f"SET SRC_LEADER = '{SRC_LEADER}'").collect()
session.sql(f"SET SIM_BAR = {int(SIM_BAR)}").collect()      # used by the SQL cells as $SIM_BAR
session.sql(f"SET NEAR_EXACT_BAR = {int(NEAR_EXACT_BAR)}").collect()   # used as $NEAR_EXACT_BAR

def n_rows(table):
    return session.sql(f"SELECT COUNT(*) AS N FROM {table}").collect()[0]["N"]

SRC_COUNTS_BEFORE = {SRC_COSMOS: n_rows(SRC_COSMOS), SRC_LEADER: n_rows(SRC_LEADER)}
print("schema:", session.get_current_schema())
print("source rows before run:", SRC_COUNTS_BEFORE)

## 1 · Existing V17 title/role guard logic, versioned for V18

In [ ]:
%%sql -r v18_udf_clean_title
CREATE OR REPLACE FUNCTION LNK_V18_CLEAN_TITLE(S VARCHAR)
  RETURNS VARCHAR
  LANGUAGE PYTHON
  RUNTIME_VERSION = '3.11'
  HANDLER = 'clean_title'
AS
$$
import re
import unicodedata

_GENDER = re.compile(r"\(\s*[mfwdhx](\s*/\s*[mfwdhx])+\s*\)|\b[mfwdhx](\s*/\s*[mfwdhx])+\b")
_DATES = re.compile(r"\b\d{1,4}[/\-.]\d{1,2}([/\-.]\d{1,4})?\b")
_NOISE = {"remote", "hybrid", "urgent"}
_FILLER = {"of", "and", "the", "for"}
_EXPAND = {
    "svp": "senior vice president", "evp": "executive vice president",
    "avp": "assistant vice president", "vp": "vice president", "sr": "senior", "snr": "senior",
    "ceo": "chief executive officer", "cfo": "chief financial officer",
    "coo": "chief operating officer", "cto": "chief technology officer",
    "cio": "chief information officer", "chro": "chief human resources officer",
    "ciso": "chief information security officer", "gc": "general counsel",
    "caio": "chief artificial intelligence officer", "cdao": "chief data analytics officer",
    "tech": "technology", "ops": "operations", "hr": "human resources", "fin": "finance",
    "mktg": "marketing", "eng": "engineering",
}

def clean_title(s):
    if s is None:
        return None
    t = unicodedata.normalize("NFKD", str(s))
    t = "".join(ch for ch in t if not unicodedata.combining(ch)).lower()
    t = _GENDER.sub(" ", t)
    t = _DATES.sub(" ", t)
    t = t.replace("&", " and ")
    t = re.sub(r"[^a-z0-9]+", " ", t)
    t = re.sub(r"\bbiz dev\b", "business development", t)
    words = []
    for w in t.split():
        if w in _NOISE:
            continue
        words.extend(_EXPAND.get(w, w).split())
    words = _drop_repeated_phrases([w for w in words if w not in _FILLER])
    out = " ".join(words)
    return out or None


def _drop_repeated_phrases(words):
    """Keep only the first occurrence of any repeated phrase of 2+ words.
    'Chief Financial Officer (CFO)' expands to 'chief financial officer chief financial officer';
    this returns 'chief financial officer'. Single repeated words are kept."""
    out = []
    i = 0
    while i < len(words):
        skipped = False
        for n in range(min(6, len(words) - i), 1, -1):
            phrase = words[i:i + n]
            for j in range(0, len(out) - n + 1):
                if out[j:j + n] == phrase:
                    i += n
                    skipped = True
                    break
            if skipped:
                break
        if not skipped:
            out.append(words[i])
            i += 1
    return out
$$

In [ ]:
%%sql -r title_extraction_udf
CREATE OR REPLACE FUNCTION LNK_V18_EXTRACT_ROLE_TITLE(JOB_TITLE VARCHAR, EMPLOYER VARCHAR)
  RETURNS VARCHAR
  LANGUAGE PYTHON
  RUNTIME_VERSION = '3.11'
  HANDLER = 'extract_role_title'
AS
$$
import re, unicodedata

def norm(s):
    s = unicodedata.normalize('NFKD', (s or '').lower())
    s = ''.join(x for x in s if not unicodedata.combining(x))
    s = re.sub(r'[^a-z0-9]+', ' ', s)
    s = re.sub(r'\b(inc|incorporated|ltd|limited|llc|lp|plc|corp|corporation|company|co|group|holdings)\b', ' ', s)
    return ' '.join(s.split())

def is_employer(s, employer):
    a, b = norm(s), norm(employer)
    if not a or not b:
        return False
    if a == b:
        return True
    # Require a genuinely recognizable company prefix, never a one-word generic fragment.
    if min(len(a), len(b)) >= 8 and (a in b or b in a):
        return True
    aw,bw=set(a.split()),set(b.split())
    return len(aw & bw) >= 2 and len(aw & bw) / max(1, min(len(aw),len(bw))) >= .75

def extract_role_title(job_title, employer):
    if job_title is None:
        return None
    title = str(job_title).strip()
    if not title:
        return title
    # e.g. 'Brookfield Asset Management, Inc. is hiring: SVP - Head of Content Strategy'.
    m = re.search(r'\b(?:is|are)\s+(?:now\s+)?hiring\s*[:\-–—]?\s+', title, re.I)
    if m and 2 <= m.start() <= 180 and len(title[m.end():].strip()) >= 4:
        prefix = title[:m.start()].strip()
        if is_employer(prefix, employer):
            return title[m.end():].strip()
    # e.g. 'SVP - Head of Content Strategy Job at Brookfield Asset Management Inc. in N'.
    # Only cut off the trailing job-board attribution when its company matches the cohort employer.
    job_at = re.search(r'\s+job\s+at\s+(.+)$', title, re.I)
    if job_at and len(title[:job_at.start()].strip()) >= 4 and is_employer(job_at.group(1), employer):
        return title[:job_at.start()].strip()
    # 'Employer | CFO' / 'Employer - CFO' only when the exact leading phrase is the employer.
    sep = re.search(r'\s+\|\s+|\s+[\-–—]\s+|:\s+', title)
    if sep and 2 <= sep.start() <= 100:
        prefix, tail = title[:sep.start()].strip(), title[sep.end():].strip()
        if len(tail) >= 4 and is_employer(prefix, employer):
            return tail
    return title
$$

In [ ]:
%%sql -r v18_udf_title_profile
CREATE OR REPLACE FUNCTION LNK_V18_TITLE_PROFILE(ORIGINAL_TITLE VARCHAR, CLEAN_TITLE VARCHAR)
  RETURNS VARIANT
  LANGUAGE PYTHON
  RUNTIME_VERSION = '3.11'
  HANDLER = 'title_profile'
AS
$$
import re

# Role level of a title, read from the CLEANED title (VP -> vice president, CFO -> chief financial
# officer ...). Longest phrases are checked first, so 'senior vice president' is never read as
# 'vice president'. Regional / area / corporate VP count as VICE_PRESIDENT.
_LEVELS = [
    ("ASSOCIATE_VICE_PRESIDENT", r"\b(associate|assistant) vice president\b"),
    ("EXECUTIVE_VICE_PRESIDENT", r"\bexecutive vice president\b"),
    ("SENIOR_VICE_PRESIDENT",    r"\bsenior vice president\b"),
    ("VICE_PRESIDENT",           r"\bvice president\b"),
    ("PRESIDENT",                r"\bpresident\b"),
    ("SENIOR_DIRECTOR",          r"\bsenior director\b"),
    ("MANAGING_DIRECTOR",        r"\bmanaging director\b"),
    ("DIRECTOR",                 r"\bdirector\b"),
    ("HEAD",                     r"\bhead\b"),
]
_CHIEF = re.compile(r"\bchief((?: [a-z]+){1,4}?) officer\b|\bchief (executive)\b")
# Standard chief-role codes, so different wordings of the same role compare equal.
# Key = the words between 'chief' and 'officer' after cleaning (filler words already removed).
_CHIEF_CODE = {
    "executive": "CEO",
    "financial": "CFO", "finance": "CFO",
    "operating": "COO", "operations": "COO", "operation": "COO",
    "technology": "CTO", "technical": "CTO",
    "information": "CIO", "information technology": "CIO",
    "information security": "CISO",
    "human resources": "CHRO", "human resource": "CHRO", "people": "CHRO",
    "people culture": "CHRO", "personnel": "CHRO",
    "artificial intelligence": "CAIO", "ai": "CAIO",
    "data analytics": "CDAO", "data ai": "CDAO", "data artificial intelligence": "CDAO",
    "administrative": "CADMIN", "administration": "CADMIN",
    "communications": "CCOMMS", "communication": "CCOMMS",
    "investment": "CINVEST", "investments": "CINVEST",   # never the same as CIO (information)
}
# 'Assistant to the CEO', 'Chief of Staff to the CFO', 'Office of the CTO': checked on the ORIGINAL
# title, because cleaning drops 'of' and 'the'
_ACRONYMS = r"(ceo|cfo|coo|cto|cio|cmo|cpo|cdo|cso|cco|chro|ciso|cro|clo|cao|cno|caio|cdao)"
_TO_OF_CSUITE = re.compile(r"\b(to|of|for)\s+(the\s+)?((group|global)\s+)?(" + _ACRONYMS + r"\b|chief\b)")


def _chief_roles(clean):
    roles = set()
    for m in _CHIEF.finditer(clean):
        inner = (m.group(1) or m.group(2) or "").strip()
        if not inner:
            continue
        # known role -> standard code; otherwise a stemmed key (first 6 letters of each word)
        roles.add(_CHIEF_CODE.get(inner, " ".join(w[:6] for w in inner.split())))
    return sorted(roles)


def title_profile(original, clean):
    """{'level': ..., 'chief_roles': [...], 'is_global': bool, 'to_of_csuite': bool}
    level is None when the title has no role level, or when it works 'to / of' a C-suite officer."""
    if not clean:
        return {"level": None, "chief_roles": [], "is_global": False, "to_of_csuite": False}
    orig = (original or "").lower()
    to_of = bool(_TO_OF_CSUITE.search(orig))
    roles = _chief_roles(clean)
    level = None
    if not to_of:
        if roles:
            level = "C_SUITE"
        else:
            for name, pat in _LEVELS:
                if re.search(pat, clean):
                    level = name
                    break
    return {"level": level, "chief_roles": roles,
            "is_global": "global" in clean.split(), "to_of_csuite": to_of}
$$

In [ ]:
%%sql -r role_context_udf
CREATE OR REPLACE FUNCTION LNK_V18_ROLE_CONTEXT(ROLE_TITLE VARCHAR, JOB_DESCRIPTION VARCHAR)
  RETURNS VARIANT
  LANGUAGE PYTHON
  RUNTIME_VERSION = '3.11'
  HANDLER = 'role_context'
AS
$$
import re

_CHIEF_ACRONYMS = re.compile(r'\b(?:ceo|cfo|coo|cto|cio|cmo|cpo|cdo|cso|cco|chro|ciso|cro|clo|cao|cno|caio|cdao)\b', re.I)
_CHIEF_PHRASE = re.compile(r'\bchief\s+(?:[a-z]+\s+){1,4}?officer\b|\bchief executive\b', re.I)
_OFFICE_NEAR = re.compile(
    r'\b(?:ceo|cfo|coo|cto|cio|cmo|cpo|chro|ciso|cro|clo|caio|cdao)\b\s*'
    r'(?:[-/&,:]|\s)*(?:office|team|department|function|division|unit|organization|support|strategy|oversight|enablement|initiatives|business management)\b'
    r'|\b(?:office|team|department|function|division|unit|organization|support|strategy|business management)'
    r'\s+(?:(?:of|for|to|in)\s+)?(?:the\s+)?(?:ceo|cfo|coo|cto|cio|cmo|cpo|chro|ciso|cro|clo|caio|cdao)\b',re.I)
_STRONG_SUPPORT = re.compile(r'\b(?:chief of staff|executive assistant|administrative assistant)\b',re.I)
_SUBORDINATE = re.compile(r'\b(?:vp|svp|avp|vice president|senior vice president|associate vice president|analyst|associate|senior professional|director|manager|specialist|coordinator|intern|trainee)\b',re.I)
_DUAL_EXECUTIVE = re.compile(r'\b(?:vice president|vp|president)\s+(?:and|&)\s+(?:chief\b|ceo\b|cfo\b|coo\b|cto\b|cio\b)',re.I)
# Only a strong hiring-for-a-junior-role statement is used; incidental analyst mentions do not count.
_JD_JUNIOR = re.compile(
    r'\b(?:we\s+are\s+looking\s+for|we\s+seek|is\s+seeking|team\s+is\s+seeking|'
    r'we\s+are\s+hiring|is\s+hiring|role\s+is\s+for|position\s+is\s+for|'
    r'looking\s+for|seeking|hiring)\b.{0,65}?\b(?:an?\s+|the\s+)?'
    r'(?:analyst|analysts|associate|associates|senior associate|senior associates|coordinator|coordinators|'
    r'junior professional|junior professionals)\b', re.I|re.S)

_JD_VP_HIRE = re.compile(
    r'\b(?:is seeking|we are seeking|we are looking for|is hiring|we are hiring|seeking|hiring)' 
    r'\b.{0,25}?\b(?:a\s+)?(?:vice president|vp)\b',re.I|re.S)
_JD_CHIEF_HIRE = re.compile(
    r'\b(?:is seeking|we are seeking|we are looking for|is hiring|we are hiring|seeking|hiring)' 
    r'\b.{0,28}?\b(?:a\s+)?(?:chief\s+(?:[a-z]+\s+){1,4}?officer|ceo|cfo|coo|cto|cio)\b',re.I|re.S)
_AMBIGUOUS_EXEC_GRADE = re.compile(r'^(?:director|senior vice president|executive vice president)\s*[,/&-]\s*(?:coo|cfo|cto|cio|ceo)\b|\b(?:field cto|field cfo|field coo)\b',re.I)

def role_context(role_title, job_description):
    title = ' '.join(str(role_title or '').lower().split())
    jd = re.sub('<[^>]+>', ' ', str(job_description or '')[:7000]); jd=' '.join(jd.split())
    has_chief = bool(_CHIEF_ACRONYMS.search(title) or _CHIEF_PHRASE.search(title))
    jd_junior = bool(_JD_JUNIOR.search(jd)) if jd else False
    jd_vp = bool(_JD_VP_HIRE.search(jd)) if jd else False
    jd_exec = bool(_JD_CHIEF_HIRE.search(jd)) if jd else False
    if not has_chief:
        return {'kind':'OTHER','reason':'NO_CHIEF_ROLE','jd_junior_signal':jd_junior}
    if _STRONG_SUPPORT.search(title) or _OFFICE_NEAR.search(title):
        return {'kind':'EXECUTIVE_OFFICE','reason':'OFFICE_TEAM_IN_TITLE','jd_junior_signal':jd_junior}
    # A detailed JD directly hiring a COO/CFO supersedes ambiguous 'VP, COO' headline metadata.
    if jd_exec:
        return {'kind':'EXECUTIVE','reason':'JD_EXPLICIT_CHIEF_HIRING','jd_junior_signal':jd_junior}
    if _SUBORDINATE.search(title) and not (_CHIEF_PHRASE.search(title) or _DUAL_EXECUTIVE.search(title)):
        if _AMBIGUOUS_EXEC_GRADE.search(title):
            return {'kind':'UNCERTAIN','reason':'AMBIGUOUS_DUAL_GRADE','jd_junior_signal':jd_junior}
        return {'kind':'EXECUTIVE_OFFICE','reason':'SUBORDINATE_GRADE_IN_TITLE','jd_junior_signal':jd_junior}
    # Explicit vacancy-grade statements: not incidental mentions of analysts/VPs elsewhere.
    if (jd_junior or jd_vp) and not _CHIEF_PHRASE.search(title):
        return {'kind':'EXECUTIVE_OFFICE','reason':'JD_EXPLICIT_NON_EXEC_HIRING','jd_junior_signal':jd_junior}
    return {'kind':'EXECUTIVE','reason':'CHIEF_ROLE_TITLE','jd_junior_signal':jd_junior}
$$

In [ ]:
%%sql -r v18_udf_title_conflict
CREATE OR REPLACE FUNCTION LNK_V18_TITLE_CONFLICT(POSTING_CLEAN VARCHAR, LEADER_CLEAN VARCHAR)
  RETURNS BOOLEAN
  LANGUAGE PYTHON
  RUNTIME_VERSION = '3.11'
  HANDLER = 'title_conflict'
AS
$$
import re

# Applied only to non-exact pairs that passed the title test (95+ similarity or word match).
# 1. Junior / individual-contributor word on one side only
_JUNIOR = {"intern", "internship", "trainee", "training", "assistant", "associate", "deputy",
           "analyst", "coordinator", "junior", "apprentice", "student", "stage", "stagiaire",
           "praktikant", "werkstudent", "rotational", "graduate",
           "accountant", "reviewer", "architect", "architekt", "engineer", "developer",
           "specialist", "consultant", "buyer", "ambassador", "controller", "auditor",
           "representative", "agent", "clerk", "administrator", "teammate", "expert", "experte",
           "advisor", "staff", "trader", "owner", "appraiser", "wholesaler",
           # non-English and placement words
           "assistenz", "assistent", "assistente", "technicien", "technician", "residence"}
# Country-level contradictions are stronger than macro-region comparisons.
# 'Taiwan' vs 'Italia' must NOT survive simply because both carry a chief-title acronym.
_COUNTRY_PATTERNS = {
    'taiwan': r'\btaiwan\b', 'italy': r'\b(?:italy|italia)\b',
    'germany': r'\b(?:germany|deutschland)\b', 'france': r'\b(?:france|francia)\b',
    'india': r'\bindia\b', 'china': r'\bchina\b', 'japan': r'\bjapan\b',
    'hong_kong': r'\bhong\s+kong\b', 'singapore': r'\bsingapore\b',
    'united_states': r'\b(?:usa|united states|united states america)\b',
    'united_kingdom': r'\b(?:uk|united kingdom|great britain)\b',
    'canada': r'\bcanada\b', 'australia': r'\baustralia\b',
    'spain': r'\b(?:spain|espana)\b', 'brazil': r'\b(?:brazil|brasil)\b',
    'mexico': r'\bmexico\b', 'switzerland': r'\b(?:switzerland|schweiz|suisse)\b',
    'netherlands': r'\b(?:netherlands|nederland)\b',
    'south_korea': r'\b(?:south korea|republic korea)\b',
    'united_arab_emirates': r'\b(?:uae|united arab emirates)\b',
}

def _named_countries(text):
    return {k for k, pattern in _COUNTRY_PATTERNS.items() if re.search(pattern, text)}

# 2. Regions: both sides name a region and they differ (one side only is fine)
_REGION = {"west", "east", "north", "south", "central", "midwest", "northeast", "southeast",
           "southwest", "northwest", "emea", "apac", "asia", "asean", "latam", "americas", "europe",
           "mena", "anz", "us", "usa", "uk", "canada", "india", "japan", "china", "germany", "france",
           "pacific", "atlantic", "gulf", "nordics", "benelux", "dach", "iberia", "africa"}
_AREA = {"apac": "APAC", "asia": "APAC", "asean": "APAC", "anz": "APAC", "pacific": "APAC",
         "japan": "APAC", "china": "APAC", "india": "APAC",
         "emea": "EMEA", "europe": "EMEA", "uk": "EMEA", "mena": "EMEA", "africa": "EMEA",
         "nordics": "EMEA", "benelux": "EMEA", "dach": "EMEA", "iberia": "EMEA",
         "germany": "EMEA", "france": "EMEA",
         "americas": "AMER", "us": "AMER", "usa": "AMER", "canada": "AMER", "latam": "AMER"}
# 3 + 4. Generic vs specific, and a shared role word: words that only say level or scope
_LEVEL = {"senior", "executive", "vice", "president", "chief", "officer", "director", "head",
          "managing", "general", "manager", "regional", "area", "global", "group", "lead", "leader",
          "assistant", "associate", "deputy", "interim", "acting", "co", "principal", "partner",
          "national", "divisional", "district", "corporate", "first"}
_CHIEF = re.compile(r"\bchief(?: [a-z]+){1,4}? officer\b|\bchief executive\b")


def title_conflict(posting_clean, leader_clean):
    """TRUE when two cleaned titles that look alike describe different roles."""
    if not posting_clean or not leader_clean:
        return False
    sa, sb = set(posting_clean.split()), set(leader_clean.split())
    if (sa & _JUNIOR) != (sb & _JUNIOR):
        return True
    # Explicitly different named countries are incompatible, even within one macro-region.
    ca_named, cb_named = _named_countries(posting_clean), _named_countries(leader_clean)
    if ca_named and cb_named and not (ca_named & cb_named):
        return True
    ra, rb = sa & _REGION, sb & _REGION
    if ra and rb and not (ra & rb):
        if not ({_AREA.get(w, w) for w in ra} & {_AREA.get(w, w) for w in rb}):
            return True
    # not applied to C-suite titles: there the chief role itself is compared
    if not (_CHIEF.search(posting_clean) or _CHIEF.search(leader_clean)):
        ca, cb = _content(sa), _content(sb)
        # generic vs specific: 'senior vice president' vs 'senior vice president growth'
        if bool(ca) != bool(cb):
            return True
        # both name a function: they must share at least one role word
        # ('svp paid social' vs 'svp ai ml' fails; 'svp engineering' vs 'svp data engineering' passes)
        if ca and cb and not (ca & cb):
            return True
    return False


def _content(words):
    """Role words: everything except level / scope and region words; plural 's' ignored."""
    return {w[:-1] if len(w) > 3 and w.endswith("s") else w for w in words - _LEVEL - _REGION}
$$

In [ ]:
%%sql -r v18_precision_smoke_tests
WITH tests AS (
    SELECT 'strip_company_prefix' AS TEST,
           LNK_V18_EXTRACT_ROLE_TITLE('Brookfield Asset Management, Inc. is hiring: SVP - GCG Head of Content Strategy', 'Brookfield Asset Management, Inc.') = 'SVP - GCG Head of Content Strategy' AS OK
    UNION ALL SELECT 'preserve_real_title',
           LNK_V18_EXTRACT_ROLE_TITLE('Chief Operating Officer - Blackstone Miami','Blackstone') = 'Chief Operating Officer - Blackstone Miami'
    UNION ALL SELECT 'strip_trailing_job_at',
           LNK_V18_EXTRACT_ROLE_TITLE('SVP - GCG Head of Content Strategy Job at Brookfield Asset Management, Inc. in N', 'Brookfield Corp.') = 'SVP - GCG Head of Content Strategy'
    UNION ALL SELECT 'coo_team_is_office',
           LNK_V18_ROLE_CONTEXT('COO Team', NULL):kind::STRING = 'EXECUTIVE_OFFICE'
    UNION ALL SELECT 'vp_in_coo_office',
           LNK_V18_ROLE_CONTEXT('FSIG COO VP - Technology Strategy', NULL):kind::STRING = 'EXECUTIVE_OFFICE'
    UNION ALL SELECT 'real_coo_is_executive',
           LNK_V18_ROLE_CONTEXT('Chief Operating Officer (COO), FMC', NULL):kind::STRING = 'EXECUTIVE'
    UNION ALL SELECT 'jd_detects_analyst_hiring',
           LNK_V18_ROLE_CONTEXT('COO - Digital, Data & Enablement', 'We are looking for analysts and associates to join our team.'):kind::STRING = 'EXECUTIVE_OFFICE'
    UNION ALL SELECT 'taiwan_italia_conflict',
           LNK_V18_TITLE_CONFLICT('chief operating officer taiwan', 'chief operating officer italia')
    UNION ALL SELECT 'taiwan_taiwan_allowed',
           NOT LNK_V18_TITLE_CONFLICT('chief operating officer taiwan', 'chief operating officer taiwan')
)
SELECT * FROM tests ORDER BY TEST

In [ ]:
%%sql -r v18_udf_smoke_test
-- smoke test: all five SVP variants must clean to the same string
SELECT t AS TITLE, LNK_V18_CLEAN_TITLE(t) AS CLEAN
FROM VALUES ('SVP Technology'), ('Senior VP Technology'), ('Senior Vice-President – Technology'),
            ('Sr. VP, Technology'), ('SVP of Technology'), ('CFO'), ('Vice President, CTO'),
            ('Deputy CEO'), ('Chief of Staff'), ('NED'), ('Assistant Vice President, Legal'), ('CMO'),
            ('Chief Legal Counsel For Europe'), ('Associate General Counsel'),
            ('Chief Financial Officer (CFO)'), ('CFO - Chief Financial Officer') AS v(t)

In [ ]:
%%sql -r v18_udf_guard_smoke_test
-- guard smoke test: EXPECTED column shows the right answer for each pair
SELECT a AS POSTING_TITLE, b AS LEADER_TITLE, expected AS EXPECTED,
       LNK_V18_TITLE_CONFLICT(LNK_V18_CLEAN_TITLE(a), LNK_V18_CLEAN_TITLE(b)) AS CONFLICT
FROM VALUES ('CTO', 'CTO Intern', TRUE), ('CISO', 'CISO Deputy', TRUE), ('CFO', 'CFO Intern', TRUE),
            ('CFO', 'Fractional CFO | Consultant', TRUE),
            ('Regional Vice President - Northeast', 'Regional Vice President - Southeast', TRUE),
            ('SVP', 'SVP Growth', TRUE),
            ('Regional Vice President - Northeast', 'Regional Vice President', FALSE),
            ('CEO', 'President & CEO', FALSE),
            ('SVP Mktg', 'SVP Mktg and Sales', FALSE),
            ('SVP, Paid Social', 'SVP, AI/ML', TRUE),
            ('SVP Engineering', 'SVP Data Engineering', FALSE),
            ('Assistenz Vorstand COO', 'COO', TRUE),
            ('CFO', 'Interim CFO', FALSE) AS v(a, b, expected)

In [ ]:
%%sql -r v18_udf_profile_smoke_test
-- profile smoke test: role level, chief role, global flag, 'to / of C-suite'
SELECT t AS TITLE, LNK_V18_CLEAN_TITLE(t) AS CLEAN, expected AS EXPECTED_LEVEL,
       LNK_V18_TITLE_PROFILE(t, LNK_V18_CLEAN_TITLE(t)):level::STRING AS LEVEL,
       LNK_V18_TITLE_PROFILE(t, LNK_V18_CLEAN_TITLE(t)):chief_roles   AS CHIEF_ROLES,
       LNK_V18_TITLE_PROFILE(t, LNK_V18_CLEAN_TITLE(t)):is_global     AS IS_GLOBAL
FROM VALUES ('VP Finance', 'VICE_PRESIDENT'), ('SVP Marketing and Finance', 'SENIOR_VICE_PRESIDENT'),
            ('EVP Sales', 'EXECUTIVE_VICE_PRESIDENT'), ('AVP Legal', 'ASSOCIATE_VICE_PRESIDENT'),
            ('Regional VP Sales', 'VICE_PRESIDENT'), ('Global VP Finance', 'VICE_PRESIDENT'),
            ('President & CEO', 'C_SUITE'), ('VP & CFO', 'C_SUITE'),
            ('Assistant to the CEO', NULL), ('Chief of Staff to the CFO', NULL), ('Office of the CFO', NULL),
            ('Executive Communications for the CMO', NULL), ('Office of the CAIO', NULL),
            ('Senior Director Finance', 'SENIOR_DIRECTOR'), ('Head of Sales', 'HEAD'),
            ('Business Analyst', NULL) AS v(t, expected)

In [ ]:
%%sql -r v18_udf_chief_synonym_smoke_test
-- chief-role synonyms: SAME_ROLE should equal EXPECTED
SELECT a AS TITLE_A, b AS TITLE_B, expected AS EXPECTED,
       LNK_V18_TITLE_PROFILE(a, LNK_V18_CLEAN_TITLE(a)):chief_roles AS ROLES_A,
       LNK_V18_TITLE_PROFILE(b, LNK_V18_CLEAN_TITLE(b)):chief_roles AS ROLES_B,
       TO_JSON(ROLES_A) = TO_JSON(ROLES_B)                  AS SAME_ROLE
FROM VALUES ('Chief People Officer', 'Chief Human Resources Officer', TRUE),
            ('Chief People & Culture Officer', 'CHRO', TRUE),
            ('CAIO', 'Chief AI Officer', TRUE),
            ('Chief Artificial Intelligence Officer', 'CAIO', TRUE),
            ('Chief Data & Analytics Officer', 'Chief Data and AI Officer', TRUE),
            ('Chief Operation Officer', 'COO', TRUE),
            ('Chief Investment Officer', 'CIO', FALSE),
            ('Chief Information Security Officer', 'CIO', FALSE),
            ('Chief Revenue Officer', 'Chief Risk Officer', FALSE) AS v(a, b, expected)

## 2 · Stage 1 — canonical Cosmos postings and normalized titles

In [ ]:
%%sql -r v18_stage1_extracted_titles
-- Step A: deduplicate, then strip *verified* employer advertisement prefixes.
CREATE OR REPLACE TABLE LNK_V18_S1_TITLE_EXTRACTED AS
WITH src AS (
    SELECT c.*
    FROM IDENTIFIER($SRC_COSMOS) c
    QUALIFY ROW_NUMBER() OVER (PARTITION BY c.CANONICAL_JOB_ID
                               ORDER BY c.MERGE_GROUP_START_DATE NULLS LAST, c.POST_DATE NULLS LAST) = 1
)
SELECT src.*,
       LNK_V18_EXTRACT_ROLE_TITLE(src.TITLE_RAW, src.COMPANY)        AS TITLE_RAW_ROLE,
       LNK_V18_EXTRACT_ROLE_TITLE(src.TITLE_TRANSLATED, src.COMPANY) AS TITLE_TRANSLATED_ROLE
FROM src

In [ ]:
%%sql -r v18_stage1_table1
-- Step B: materialize normalized titles; do not chain Python UDFs in one SQL query.
CREATE OR REPLACE TABLE LNK_V18_T1_COSMOS AS
WITH cleaned AS (
    SELECT src.*,
           DATE_TRUNC('month', COALESCE(TRY_TO_DATE(LEFT(TO_VARCHAR(src.MERGE_GROUP_START_DATE), 10)),
                                        TRY_TO_DATE(LEFT(TO_VARCHAR(src.POST_DATE), 10)))) AS POSTING_MONTH,
           LNK_V18_CLEAN_TITLE(src.TITLE_RAW_ROLE)        AS TITLE_RAW_CLEAN,
           LNK_V18_CLEAN_TITLE(src.TITLE_TRANSLATED_ROLE) AS TITLE_TRANSLATED_CLEAN
    FROM LNK_V18_S1_TITLE_EXTRACTED src
)
SELECT cleaned.*,
       TRY_TO_NUMBER(TO_VARCHAR(RCID)) AS RCID_NUM,
       ARRAY_DISTINCT(ARRAY_CAT(
           COALESCE(STRTOK_TO_ARRAY(UPPER(REGEXP_REPLACE(ALL_COUNTRY_POSTED, '\\s*,\\s*', ',')), ','), ARRAY_CONSTRUCT()),
           COALESCE(ARRAY_CONSTRUCT_COMPACT(UPPER(TRIM(COUNTRY))), ARRAY_CONSTRUCT())
       )) AS COUNTRIES_POSTED,
       COALESCE(TRY_TO_DATE(LEFT(TO_VARCHAR(POST_DATE), 10)),
                TRY_TO_DATE(LEFT(TO_VARCHAR(CANONICAL_POST_DATE), 10))) AS ACTUAL_POST_DATE,
       IFF(TITLE_RAW_ROLE IS DISTINCT FROM TITLE_RAW, TRUE, FALSE) AS RAW_EMPLOYER_PREFIX_REMOVED,
       IFF(TITLE_TRANSLATED_ROLE IS DISTINCT FROM TITLE_TRANSLATED, TRUE, FALSE) AS TR_EMPLOYER_PREFIX_REMOVED
FROM cleaned

In [ ]:
%%sql -r v18_stage1_check
SELECT COUNT(*)                                        AS POSTINGS,
       COUNT(DISTINCT CANONICAL_JOB_ID)                AS DISTINCT_JOB_IDS,
       COUNT_IF(RCID_NUM IS NULL)                      AS NO_RCID,
       COUNT_IF(POSTING_MONTH IS NULL)                 AS NO_POSTING_DATE,
       COUNT_IF(TITLE_RAW_CLEAN IS NULL AND TITLE_TRANSLATED_CLEAN IS NULL) AS NO_TITLE
FROM LNK_V18_T1_COSMOS

## 3 · Stage 2 — posting / leader profiles and candidate scoring

In [ ]:
%%sql -r stage2_posting_profiles
CREATE OR REPLACE TABLE LNK_V18_S2_POSTING_PROFILES AS
SELECT CANONICAL_JOB_ID, RCID_NUM, POSTING_MONTH, TITLE_RAW, TITLE_TRANSLATED, TITLE_RAW_ROLE, TITLE_TRANSLATED_ROLE, ACTUAL_POST_DATE,
       TITLE_RAW_CLEAN, TITLE_TRANSLATED_CLEAN, COUNTRIES_POSTED,
       ARRAY_DISTINCT(SPLIT(TITLE_RAW_CLEAN, ' ')) AS RAW_WORDS,
       ARRAY_DISTINCT(SPLIT(TITLE_TRANSLATED_CLEAN, ' ')) AS TR_WORDS,
       LNK_V18_TITLE_PROFILE(TITLE_RAW_ROLE, TITLE_RAW_CLEAN) AS RAW_PROF,
       LNK_V18_TITLE_PROFILE(TITLE_TRANSLATED_ROLE, TITLE_TRANSLATED_CLEAN) AS TR_PROF
FROM LNK_V18_T1_COSMOS
WHERE RCID_NUM IS NOT NULL AND POSTING_MONTH IS NOT NULL

In [ ]:
%%sql -r v18_post_role_context
CREATE OR REPLACE TABLE LNK_V18_S2_POSTING_ROLE_CONTEXT AS
SELECT CANONICAL_JOB_ID,
       LNK_V18_ROLE_CONTEXT(TITLE_RAW_ROLE,
          LEFT(COALESCE(SELECTED_DESCRIPTION, DESCRIPTION, ''), 7000)) AS RAW_CTX,
       LNK_V18_ROLE_CONTEXT(TITLE_TRANSLATED_ROLE,
          LEFT(COALESCE(SELECTED_DESCRIPTION, DESCRIPTION, ''), 7000)) AS TR_CTX
FROM LNK_V18_T1_COSMOS

In [ ]:
%%sql -r stage2_leader_titles_clean
CREATE OR REPLACE TABLE LNK_V18_S2_LEADER_TITLES_CLEAN AS
WITH bounds AS (
    SELECT LEAST(DATE_FROM_PARTS(YEAR(MIN(POSTING_MONTH)), 1, 1),
                 DATEADD(month, -6, MIN(POSTING_MONTH))) AS LO,
           DATEADD(month, 12, MAX(POSTING_MONTH)) AS HI
    FROM LNK_V18_S2_POSTING_PROFILES
),
titles AS (
    SELECT DISTINCT lr.SOURCE_ROLE_TITLE
    FROM IDENTIFIER($SRC_LEADER) lr, bounds b
    WHERE lr.SOURCE_ROLE_TITLE IS NOT NULL
      AND lr.REVELIO_RCID IS NOT NULL
      AND lr.ROLE_START IS NOT NULL
      AND TRY_TO_DATE(LEFT(TO_VARCHAR(lr.ROLE_START), 10)) BETWEEN b.LO AND b.HI
      AND TRY_TO_NUMBER(TO_VARCHAR(lr.REVELIO_RCID)) IN
             (SELECT DISTINCT RCID_NUM FROM LNK_V18_S2_POSTING_PROFILES)
)
SELECT SOURCE_ROLE_TITLE,
       LNK_V18_CLEAN_TITLE(SOURCE_ROLE_TITLE) AS SOURCE_ROLE_TITLE_CLEAN
FROM titles

In [ ]:
%%sql -r v18_leader_role_context
CREATE OR REPLACE TABLE LNK_V18_S2_LEADER_ROLE_CONTEXT AS
SELECT SOURCE_ROLE_TITLE,
       LNK_V18_ROLE_CONTEXT(SOURCE_ROLE_TITLE, NULL) AS L_CTX
FROM LNK_V18_S2_LEADER_TITLES_CLEAN

In [ ]:
%%sql -r v18_leader_title_profiles
CREATE OR REPLACE TABLE LNK_V18_S2_LEADER_TITLE_PROFILES AS
SELECT t.SOURCE_ROLE_TITLE, t.SOURCE_ROLE_TITLE_CLEAN,
       LNK_V18_TITLE_PROFILE(t.SOURCE_ROLE_TITLE, t.SOURCE_ROLE_TITLE_CLEAN) AS L_PROF,
       c.L_CTX
FROM LNK_V18_S2_LEADER_TITLES_CLEAN t
LEFT JOIN LNK_V18_S2_LEADER_ROLE_CONTEXT c ON c.SOURCE_ROLE_TITLE = t.SOURCE_ROLE_TITLE

In [ ]:
%%sql -r stage2_candidate_scores
CREATE OR REPLACE TABLE LNK_V18_S2_PAIR_SCORES AS
WITH p AS (
    SELECT x.*,
           COALESCE(pc.RAW_CTX:kind::STRING, 'OTHER') AS RAW_KIND,
           COALESCE(pc.TR_CTX:kind::STRING,  'OTHER') AS TR_KIND,
           COALESCE(pc.RAW_CTX:reason::STRING, '') AS RAW_CONTEXT_REASON,
           COALESCE(pc.TR_CTX:reason::STRING, '')  AS TR_CONTEXT_REASON,
           COALESCE(pc.RAW_CTX:jd_junior_signal::BOOLEAN, FALSE) AS RAW_JD_JUNIOR_SIGNAL,
           COALESCE(pc.TR_CTX:jd_junior_signal::BOOLEAN, FALSE)  AS TR_JD_JUNIOR_SIGNAL,
           x.RAW_PROF:level::STRING AS RAW_LEVEL,
           x.TR_PROF:level::STRING AS TR_LEVEL,
           COALESCE(x.RAW_PROF:chief_roles::ARRAY, ARRAY_CONSTRUCT()) AS RAW_ROLES,
           COALESCE(x.TR_PROF:chief_roles::ARRAY, ARRAY_CONSTRUCT()) AS TR_ROLES,
           COALESCE(x.RAW_PROF:is_global::BOOLEAN, FALSE) AS RAW_GLOBAL,
           COALESCE(x.TR_PROF:is_global::BOOLEAN, FALSE) AS TR_GLOBAL
    FROM LNK_V18_S2_POSTING_PROFILES x
    LEFT JOIN LNK_V18_S2_POSTING_ROLE_CONTEXT pc ON pc.CANONICAL_JOB_ID = x.CANONICAL_JOB_ID
),
lv AS (   -- seniority-only words: a title made only of these cannot word-match
    SELECT ARRAY_CONSTRUCT('senior', 'executive', 'vice', 'president', 'chief', 'officer', 'director',
                           'head', 'managing', 'general', 'manager', 'regional', 'area', 'global', 'group',
                           'lead', 'leader', 'assistant', 'associate', 'deputy', 'interim', 'acting', 'co',
                           'principal', 'partner', 'national', 'divisional', 'district', 'corporate',
                           'first') AS LEVEL_WORDS
),
bounds AS (
    SELECT LEAST(DATE_FROM_PARTS(YEAR(MIN(POSTING_MONTH)), 1, 1),
                 DATEADD(month, -6, MIN(POSTING_MONTH)))  AS LO,
           DATEADD(month, 12, MAX(POSTING_MONTH))         AS HI
    FROM p
),
l AS (   -- Leader Record read in place: posting RCIDs and the overall date range only
    SELECT lr.PERSON_ID, lr.CLUS_ID, lr.REVELIO_RCID,
           TRY_TO_NUMBER(TO_VARCHAR(lr.REVELIO_RCID)) AS L_RCID,
           lr.COMPANY_NAME, lr.SOURCE_ROLE_TITLE,
           TRY_TO_DATE(LEFT(TO_VARCHAR(lr.ROLE_START), 10)) AS ROLE_START,
           lr.PERSON_COUNTRY
    FROM IDENTIFIER($SRC_LEADER) lr, bounds b
    WHERE lr.REVELIO_RCID IS NOT NULL
      AND lr.ROLE_START IS NOT NULL
      AND TRY_TO_DATE(LEFT(TO_VARCHAR(lr.ROLE_START), 10)) BETWEEN b.LO AND b.HI
      AND TRY_TO_NUMBER(TO_VARCHAR(lr.REVELIO_RCID)) IN (SELECT DISTINCT RCID_NUM FROM p)
),
pairs AS (   -- employer gate + start-date gate
    SELECT p.*, l.PERSON_ID, l.CLUS_ID, l.REVELIO_RCID, l.COMPANY_NAME, l.SOURCE_ROLE_TITLE,
           l.ROLE_START, l.PERSON_COUNTRY
    FROM p
    JOIN l
      ON l.L_RCID = p.RCID_NUM
     AND (   l.ROLE_START BETWEEN DATEADD(month, -6, p.POSTING_MONTH) AND DATEADD(month, 12, p.POSTING_MONTH)
          OR l.ROLE_START = DATE_FROM_PARTS(YEAR(p.POSTING_MONTH), 1, 1))
),
leader_titles AS (
    SELECT t.SOURCE_ROLE_TITLE, t.SOURCE_ROLE_TITLE_CLEAN,
           ARRAY_DISTINCT(SPLIT(t.SOURCE_ROLE_TITLE_CLEAN, ' ')) AS L_WORDS,
           t.L_PROF:level::STRING AS L_LEVEL,
           COALESCE(t.L_PROF:chief_roles::ARRAY, ARRAY_CONSTRUCT()) AS L_ROLES,
           COALESCE(t.L_PROF:is_global::BOOLEAN, FALSE) AS L_GLOBAL,
           COALESCE(t.L_CTX:kind::STRING, 'OTHER') AS L_KIND,
           COALESCE(t.L_CTX:reason::STRING, '') AS L_CONTEXT_REASON
    FROM LNK_V18_S2_LEADER_TITLE_PROFILES t
),
scored AS (
    SELECT pr.*,
        lt.SOURCE_ROLE_TITLE_CLEAN, lt.L_LEVEL, lt.L_ROLES, lt.L_GLOBAL,
        lt.L_KIND, lt.L_CONTEXT_REASON,
        COALESCE(
          (pr.RAW_KIND = 'EXECUTIVE' AND lt.L_KIND = 'EXECUTIVE_OFFICE') OR
          (pr.RAW_KIND = 'EXECUTIVE_OFFICE' AND lt.L_KIND = 'EXECUTIVE'), FALSE
        ) AS RAW_CONTEXT_CONFLICT,
        COALESCE(
          (pr.TR_KIND = 'EXECUTIVE' AND lt.L_KIND = 'EXECUTIVE_OFFICE') OR
          (pr.TR_KIND = 'EXECUTIVE_OFFICE' AND lt.L_KIND = 'EXECUTIVE'), FALSE
        ) AS TR_CONTEXT_CONFLICT,
        -- each posting title (raw, translated) compared with the leader title on its own
        COALESCE(pr.TITLE_RAW_CLEAN = lt.SOURCE_ROLE_TITLE_CLEAN, FALSE)                          AS RAW_EXACT,
        COALESCE(pr.TITLE_TRANSLATED_CLEAN = lt.SOURCE_ROLE_TITLE_CLEAN, FALSE)                   AS TR_EXACT,
        COALESCE(JAROWINKLER_SIMILARITY(pr.TITLE_RAW_CLEAN, lt.SOURCE_ROLE_TITLE_CLEAN), 0)        AS SIM_RAW,
        COALESCE(JAROWINKLER_SIMILARITY(pr.TITLE_TRANSLATED_CLEAN, lt.SOURCE_ROLE_TITLE_CLEAN), 0) AS SIM_TRANSLATED,
        -- same chief role(s) on both sides (only matters for C_SUITE)
        COALESCE(ARRAY_SIZE(ARRAY_EXCEPT(pr.RAW_ROLES, lt.L_ROLES)) = 0
             AND ARRAY_SIZE(ARRAY_EXCEPT(lt.L_ROLES, pr.RAW_ROLES)) = 0, FALSE)                  AS RAW_SAME_ROLES,
        COALESCE(ARRAY_SIZE(ARRAY_EXCEPT(pr.TR_ROLES, lt.L_ROLES)) = 0
             AND ARRAY_SIZE(ARRAY_EXCEPT(lt.L_ROLES, pr.TR_ROLES)) = 0, FALSE)                   AS TR_SAME_ROLES,
        -- word match needs: both titles have a role level, the same level,
        -- and for C-suite the same chief role ('to / of' C-suite titles have no level)
        COALESCE(pr.RAW_LEVEL = lt.L_LEVEL
                 AND (pr.RAW_LEVEL <> 'C_SUITE' OR RAW_SAME_ROLES), FALSE)                       AS RAW_SAME_LEVEL,
        COALESCE(pr.TR_LEVEL = lt.L_LEVEL
                 AND (pr.TR_LEVEL <> 'C_SUITE' OR TR_SAME_ROLES), FALSE)                         AS TR_SAME_LEVEL,
        -- similarity keeps a level check: rejected only when both titles have a level and it differs
        -- v14: a non-exact similarity match cannot pair a C-suite role with a non-C-suite
        -- or unclassified support title. For other role levels, retain the v13 allowance.
        (COALESCE(pr.RAW_LEVEL = 'C_SUITE', FALSE) = COALESCE(lt.L_LEVEL = 'C_SUITE', FALSE))
          AND (pr.RAW_LEVEL IS NULL OR lt.L_LEVEL IS NULL
               OR (pr.RAW_LEVEL = lt.L_LEVEL AND (pr.RAW_LEVEL <> 'C_SUITE' OR RAW_SAME_ROLES))) AS RAW_LEVEL_OK,
        (COALESCE(pr.TR_LEVEL = 'C_SUITE', FALSE) = COALESCE(lt.L_LEVEL = 'C_SUITE', FALSE))
          AND (pr.TR_LEVEL IS NULL OR lt.L_LEVEL IS NULL
               OR (pr.TR_LEVEL = lt.L_LEVEL AND (pr.TR_LEVEL <> 'C_SUITE' OR TR_SAME_ROLES)))   AS TR_LEVEL_OK,
        -- word containment: every word of the shorter title appears in the other, and the shorter
        -- title has a word beyond level words (or is 'chief executive officer')
        COALESCE(
            (ARRAY_SIZE(ARRAY_EXCEPT(pr.RAW_WORDS, lt.L_WORDS)) = 0
               AND (ARRAY_SIZE(ARRAY_EXCEPT(pr.RAW_WORDS, lv.LEVEL_WORDS)) > 0
                    OR CONTAINS(pr.TITLE_RAW_CLEAN, 'chief executive officer')))
         OR (ARRAY_SIZE(ARRAY_EXCEPT(lt.L_WORDS, pr.RAW_WORDS)) = 0
               AND (ARRAY_SIZE(ARRAY_EXCEPT(lt.L_WORDS, lv.LEVEL_WORDS)) > 0
                    OR CONTAINS(lt.SOURCE_ROLE_TITLE_CLEAN, 'chief executive officer'))), FALSE)  AS RAW_CONTAINED,
        COALESCE(
            (ARRAY_SIZE(ARRAY_EXCEPT(pr.TR_WORDS, lt.L_WORDS)) = 0
               AND (ARRAY_SIZE(ARRAY_EXCEPT(pr.TR_WORDS, lv.LEVEL_WORDS)) > 0
                    OR CONTAINS(pr.TITLE_TRANSLATED_CLEAN, 'chief executive officer')))
         OR (ARRAY_SIZE(ARRAY_EXCEPT(lt.L_WORDS, pr.TR_WORDS)) = 0
               AND (ARRAY_SIZE(ARRAY_EXCEPT(lt.L_WORDS, lv.LEVEL_WORDS)) > 0
                    OR CONTAINS(lt.SOURCE_ROLE_TITLE_CLEAN, 'chief executive officer'))), FALSE)  AS TR_CONTAINED,
        RAW_SAME_LEVEL AND RAW_CONTAINED                                                         AS RAW_WORD_MATCH,
        TR_SAME_LEVEL  AND TR_CONTAINED                                                          AS TR_WORD_MATCH,
        SIM_RAW >= $SIM_BAR AND RAW_LEVEL_OK                                                     AS RAW_SIM_PASS,
        SIM_TRANSLATED >= $SIM_BAR AND TR_LEVEL_OK                                               AS TR_SIM_PASS,
        RAW_EXACT OR RAW_SIM_PASS OR RAW_WORD_MATCH                                              AS RAW_PASS,
        TR_EXACT  OR TR_SIM_PASS  OR TR_WORD_MATCH                                               AS TR_PASS,
        pr.ROLE_START >= pr.POSTING_MONTH                                                   AS IS_AFTER,
        pr.ROLE_START <= DATEADD(month, 3, pr.POSTING_MONTH)                                AS IN_3M,
        pr.ROLE_START <= DATEADD(month, 6, pr.POSTING_MONTH)                                AS IN_6M,
        pr.ROLE_START <= DATEADD(month, 9, pr.POSTING_MONTH)                                AS IN_9M,
        pr.ROLE_START <= DATEADD(month, 12, pr.POSTING_MONTH)                               AS IN_12M,
        pr.ROLE_START = DATE_FROM_PARTS(YEAR(pr.POSTING_MONTH), 1, 1)                       AS IS_JAN1,
        pr.ROLE_START <  pr.POSTING_MONTH
          AND pr.ROLE_START >= DATEADD(month, -6, pr.POSTING_MONTH)                          AS IN_6M_BEFORE
    FROM pairs pr
    LEFT JOIN leader_titles lt ON lt.SOURCE_ROLE_TITLE = pr.SOURCE_ROLE_TITLE
    CROSS JOIN lv

)
SELECT * FROM scored
WHERE RAW_PASS OR TR_PASS

In [ ]:
%%sql -r stage2_title_conflicts
CREATE OR REPLACE TABLE LNK_V18_S2_PAIR_CONFLICTS AS
SELECT s.*,
       IFF(NOT s.RAW_EXACT AND s.RAW_PASS,
           COALESCE(LNK_V18_TITLE_CONFLICT(s.TITLE_RAW_CLEAN, s.SOURCE_ROLE_TITLE_CLEAN), FALSE),
           FALSE) AS RAW_CONFLICT,
       IFF(NOT s.TR_EXACT AND s.TR_PASS,
           COALESCE(LNK_V18_TITLE_CONFLICT(s.TITLE_TRANSLATED_CLEAN, s.SOURCE_ROLE_TITLE_CLEAN), FALSE),
           FALSE) AS TR_CONFLICT
FROM LNK_V18_S2_PAIR_SCORES s

In [ ]:
%%sql -r v18_guard_diagnostic
-- Context contradictions are a separate hard gate, evaluated on both title sides.
SELECT COUNT(*) AS PAIRS_PRE_CONTEXT_GUARD,
       COALESCE(COUNT_IF(RAW_PASS AND RAW_CONTEXT_CONFLICT), 0) AS RAW_CONTEXT_CONTRADICTIONS,
       COALESCE(COUNT_IF(TR_PASS AND TR_CONTEXT_CONFLICT), 0) AS TRANSLATED_CONTEXT_CONTRADICTIONS,
       COALESCE(COUNT_IF(RAW_PASS AND NOT RAW_CONFLICT AND NOT RAW_CONTEXT_CONFLICT), 0) AS RAW_PASSES_ALL_GUARDS,
       COALESCE(COUNT_IF(TR_PASS AND NOT TR_CONFLICT AND NOT TR_CONTEXT_CONFLICT), 0) AS TR_PASSES_ALL_GUARDS
FROM LNK_V18_S2_PAIR_CONFLICTS

## 4 · Stage 3–4 — assign the eight V18 rules and one candidate per posting/person

**Critical:** exact/≥98 = Tier 1; similarity 95–97 or accepted word match = lower tier. The first matching rule is assigned using the eight-rule `CASE`. A person with multiple qualifying career-history rows keeps the row with the best **V18** ranking.

In [ ]:
%%sql -r stage2to4_candidates
CREATE OR REPLACE TABLE LNK_V18_T2_QC_BASE AS
WITH scored AS (
    SELECT s.*,
        (RAW_PASS AND NOT RAW_CONFLICT AND NOT RAW_CONTEXT_CONFLICT) AS RAW_QUALIFIES,
        (TR_PASS AND NOT TR_CONFLICT AND NOT TR_CONTEXT_CONFLICT) AS TR_QUALIFIES,
        -- Among VALID sides: EXACT > SIMILARITY >=95 > WORD; then highest Jaro-Winkler score.
        CASE WHEN RAW_EXACT AND RAW_QUALIFIES THEN 'TITLE_RAW'
             WHEN TR_EXACT AND TR_QUALIFIES THEN 'TITLE_TRANSLATED'
             WHEN RAW_SIM_PASS AND RAW_QUALIFIES
                  AND (NOT (TR_SIM_PASS AND TR_QUALIFIES) OR SIM_RAW >= SIM_TRANSLATED)
                  THEN 'TITLE_RAW'
             WHEN TR_SIM_PASS AND TR_QUALIFIES THEN 'TITLE_TRANSLATED'
             WHEN RAW_WORD_MATCH AND RAW_QUALIFIES
                  AND (NOT (TR_WORD_MATCH AND TR_QUALIFIES) OR SIM_RAW >= SIM_TRANSLATED)
                  THEN 'TITLE_RAW'
             WHEN TR_WORD_MATCH AND TR_QUALIFIES THEN 'TITLE_TRANSLATED'
             ELSE NULL END                                                                      AS TITLE_MATCHED_ON,
        TITLE_MATCHED_ON = 'TITLE_RAW'                                                          AS ON_RAW,
        (RAW_EXACT AND RAW_QUALIFIES) OR (TR_EXACT AND TR_QUALIFIES) AS TITLE_EXACT,
        IFF(ON_RAW, RAW_QUALIFIES, TR_QUALIFIES)                                               AS TITLE_PASS,
        IFF(TITLE_EXACT, 100, IFF(ON_RAW, SIM_RAW, SIM_TRANSLATED))                             AS TITLE_SIM,
        IFF(ON_RAW, TITLE_RAW_CLEAN, TITLE_TRANSLATED_CLEAN)                              AS MATCHED_POSTING_CLEAN,
        IFF(ON_RAW, RAW_LEVEL, TR_LEVEL)                                                  AS MATCHED_POSTING_LEVEL,
        CASE WHEN TITLE_EXACT THEN 'EXACT'
             WHEN IFF(ON_RAW, RAW_SIM_PASS, TR_SIM_PASS) THEN 'SIMILARITY'
             WHEN IFF(ON_RAW, RAW_WORD_MATCH, TR_WORD_MATCH) THEN 'WORD_MATCH' END              AS TITLE_MATCH_TYPE
    FROM LNK_V18_S2_PAIR_CONFLICTS s
),
guarded AS (   -- retain V17 title/context hard gates
    SELECT scored.*,
        IFF(TITLE_EXACT, FALSE, IFF(ON_RAW, RAW_CONFLICT, TR_CONFLICT)) AS TITLE_CONFLICT,
        IFF(ON_RAW, RAW_CONTEXT_CONFLICT, TR_CONTEXT_CONFLICT) AS TITLE_CONTEXT_CONFLICT,
        NOT TITLE_EXACT AND TITLE_PASS AND NOT TITLE_CONFLICT AND NOT TITLE_CONTEXT_CONFLICT AS TITLE_STRONG
    FROM scored
    WHERE TITLE_PASS
),
tiered AS (
    SELECT guarded.*,
        (TITLE_EXACT OR (TITLE_MATCH_TYPE = 'SIMILARITY' AND TITLE_SIM >= $NEAR_EXACT_BAR)) AS TITLE_TIER1,
        (TITLE_STRONG AND NOT TITLE_TIER1) AS TITLE_LOWER_TIER
    FROM guarded
),
ruled AS (
    SELECT tiered.*,
        CASE
            WHEN IS_AFTER AND IN_6M  AND TITLE_TIER1       THEN 1
            WHEN IS_JAN1              AND TITLE_TIER1       THEN 2
            WHEN IS_AFTER AND IN_6M  AND TITLE_LOWER_TIER  THEN 3
            WHEN IS_JAN1              AND TITLE_LOWER_TIER  THEN 4
            WHEN IS_AFTER AND IN_12M AND TITLE_TIER1       THEN 5
            WHEN IS_AFTER AND IN_12M AND TITLE_LOWER_TIER  THEN 6
            WHEN IN_6M_BEFORE        AND TITLE_TIER1       THEN 7
            WHEN IN_6M_BEFORE        AND TITLE_LOWER_TIER  THEN 8
        END AS RULE_NO
    FROM tiered
)
SELECT
    CANONICAL_JOB_ID, POSTING_MONTH, TITLE_RAW, TITLE_TRANSLATED, TITLE_RAW_CLEAN, TITLE_TRANSLATED_CLEAN,
    COUNTRIES_POSTED,
    PERSON_ID, CLUS_ID, REVELIO_RCID, COMPANY_NAME, SOURCE_ROLE_TITLE, SOURCE_ROLE_TITLE_CLEAN,
    ROLE_START, PERSON_COUNTRY, ACTUAL_POST_DATE,
    DATEDIFF(day, ACTUAL_POST_DATE, ROLE_START) AS POST_TO_START_DAYS,
    RAW_KIND, TR_KIND, L_KIND, RAW_CONTEXT_REASON, TR_CONTEXT_REASON, L_CONTEXT_REASON,
    RAW_JD_JUNIOR_SIGNAL, TR_JD_JUNIOR_SIGNAL,
    RAW_CONTEXT_CONFLICT, TR_CONTEXT_CONFLICT, TITLE_CONTEXT_CONFLICT,
    RULE_NO,
    CASE WHEN RULE_NO IN (1,3,5,6) THEN 1 WHEN RULE_NO IN (2,4) THEN 2 ELSE 3 END AS TIMING_CLASS, -- audit only
    CASE WHEN RULE_NO IN (1,3) THEN 6 WHEN RULE_NO IN (5,6) THEN 12
         WHEN RULE_NO IN (7,8) THEN -6 ELSE 0 END AS DATE_WINDOW, -- audit only, months
    CASE WHEN TITLE_MATCH_TYPE = 'EXACT'
           OR (TITLE_MATCH_TYPE = 'SIMILARITY' AND TITLE_SIM >= $NEAR_EXACT_BAR) THEN 1    -- exact or near-exact
         WHEN TITLE_MATCH_TYPE = 'SIMILARITY' THEN 2                                       -- similarity 95-97
         ELSE 3 END                                                                 AS MATCH_TIER,      -- word match
    TITLE_EXACT, TITLE_SIM, TITLE_MATCHED_ON, TITLE_MATCH_TYPE, TITLE_CONFLICT,
    TITLE_TIER1, TITLE_LOWER_TIER,
    -- All six match-method tests remain visible for auditing; one best valid method is ranked.
    RAW_EXACT, TR_EXACT, RAW_SIM_PASS, TR_SIM_PASS, RAW_WORD_MATCH, TR_WORD_MATCH,
    RAW_CONFLICT, TR_CONFLICT, RAW_QUALIFIES, TR_QUALIFIES, SIM_RAW, SIM_TRANSLATED,
    MATCHED_POSTING_LEVEL, L_LEVEL AS LEADER_LEVEL,
    DATEDIFF(month, POSTING_MONTH, ROLE_START)                                    AS MONTHS_TO_START,
    CASE WHEN PERSON_COUNTRY IS NULL OR ARRAY_SIZE(COUNTRIES_POSTED) = 0 THEN 'UNKNOWN'
         WHEN ARRAY_CONTAINS(UPPER(TRIM(PERSON_COUNTRY))::VARIANT, COUNTRIES_POSTED) THEN 'YES'
         ELSE 'NO' END                                                             AS COUNTRY_MATCH,
    SHA2(CONCAT_WS('|', COALESCE(TO_VARCHAR(PERSON_ID), ''), COALESCE(TO_VARCHAR(REVELIO_RCID), ''),
                   TO_VARCHAR(ROLE_START), COALESCE(SOURCE_ROLE_TITLE, '')))     AS APPOINTMENT_KEY
FROM ruled
WHERE RULE_NO IS NOT NULL
-- one row per posting x person: a person's best qualifying role for that posting
QUALIFY ROW_NUMBER() OVER (PARTITION BY CANONICAL_JOB_ID, PERSON_ID
                           ORDER BY RULE_NO, MATCH_TIER, TITLE_SIM DESC,
                                    IFF(COUNTRY_MATCH = 'YES', 0, 1),
                                    ABS(DATEDIFF(day, POSTING_MONTH, ROLE_START)),
                                    ROLE_START, CLUS_ID, APPOINTMENT_KEY) = 1

In [ ]:
%%sql -r v18_stage2to4_check
SELECT RULE_NO, COUNT(*) AS CANDIDATES, COUNT(DISTINCT CANONICAL_JOB_ID) AS POSTINGS,
       COUNT(DISTINCT PERSON_ID) AS PEOPLE
FROM LNK_V18_T2_QC_BASE
GROUP BY RULE_NO
ORDER BY RULE_NO

## 5 · Stage 5 — rank all candidates within each posting

The ranking now begins with `RULE_NO`, not `TIMING_CLASS`. There is **no `DATE_WINDOW` tie-breaker**. Similarity outranks country, and country outranks start-date proximity.

In [ ]:
%%sql -r v18_stage5_table2_qc
CREATE OR REPLACE TABLE LNK_V18_T2_QC AS
WITH keyed AS (
    SELECT q.*,
           IFF(q.COUNTRY_MATCH = 'YES', 0, 1)                AS COUNTRY_SORT,
           ABS(DATEDIFF(day, q.POSTING_MONTH, q.ROLE_START)) AS DAYS_FROM_POSTING
    FROM LNK_V18_T2_QC_BASE q
),
ranked AS (
    SELECT keyed.*,
           RANK() OVER (PARTITION BY CANONICAL_JOB_ID
                        ORDER BY RULE_NO, MATCH_TIER, TITLE_SIM DESC,
                                 COUNTRY_SORT, DAYS_FROM_POSTING)                               AS CANDIDATE_RANK,
           ROW_NUMBER() OVER (PARTITION BY CANONICAL_JOB_ID
                              ORDER BY RULE_NO, MATCH_TIER, TITLE_SIM DESC,
                                       COUNTRY_SORT, DAYS_FROM_POSTING, PERSON_ID)            AS PICK_ORDER
    FROM keyed
)
SELECT ranked.* EXCLUDE (COUNTRY_SORT, PICK_ORDER),
       PICK_ORDER = 1                                   AS IS_CHOSEN,
       CASE WHEN PICK_ORDER = 1     THEN NULL
            WHEN CANDIDATE_RANK = 1 THEN 'TIE'
            ELSE 'LOWER_RANK' END                       AS RELEASED_REASON
FROM ranked

## 6 · Stage 6 — final table and match status

In [ ]:
%%sql -r v18_stage6_table3_final
CREATE OR REPLACE TABLE LNK_V18_T3_FINAL AS
WITH per_posting AS (
    SELECT CANONICAL_JOB_ID,
           COUNT(*)                     AS CANDIDATE_COUNT,
           COUNT_IF(CANDIDATE_RANK = 1) AS N_TOP
    FROM LNK_V18_T2_QC
    GROUP BY CANONICAL_JOB_ID
)
SELECT t1.*,
       q.PERSON_ID          AS LR_PERSON_ID,
       q.SOURCE_ROLE_TITLE  AS LR_SOURCE_ROLE_TITLE,
       q.REVELIO_RCID       AS LR_REVELIO_RCID,
       q.ROLE_START         AS LR_ROLE_START,
       q.COMPANY_NAME       AS LR_COMPANY_NAME,
       q.RULE_NO            AS MATCH_RULE_NO,      -- the chosen person's own rule
       q.TITLE_MATCHED_ON   AS LR_TITLE_MATCHED_ON,
       q.TITLE_MATCH_TYPE   AS LR_TITLE_MATCH_TYPE,
       q.TITLE_SIM          AS LR_TITLE_SIM,
       q.MATCH_TIER         AS LR_MATCH_TIER,
       q.COUNTRY_MATCH      AS LR_COUNTRY_MATCH,
       q.POST_TO_START_DAYS AS LR_POST_TO_START_DAYS,
       IFF(q.TITLE_MATCHED_ON = 'TITLE_RAW', q.RAW_KIND, q.TR_KIND) AS LR_POSTING_ROLE_CONTEXT,
       q.L_KIND             AS LR_LEADER_ROLE_CONTEXT,
       ARRAY_CONSTRUCT_COMPACT(
           IFF(q.PERSON_ID IS NOT NULL AND q.POST_TO_START_DAYS < 0, 'PRE_POSTING_ROLE_START', NULL),
           IFF(q.TITLE_MATCH_TYPE = 'WORD_MATCH', 'WORD_MATCH_REVIEW', NULL),
           IFF(pp.N_TOP > 1, 'TIED_TOP_CANDIDATES', NULL),
           IFF(q.PERSON_ID IS NOT NULL AND
               IFF(q.TITLE_MATCHED_ON = 'TITLE_RAW', q.RAW_JD_JUNIOR_SIGNAL, q.TR_JD_JUNIOR_SIGNAL),
               'JD_JUNIOR_ROLE_SIGNAL', NULL),
           IFF(q.PERSON_ID IS NOT NULL AND
               (IFF(q.TITLE_MATCHED_ON = 'TITLE_RAW', q.RAW_KIND, q.TR_KIND) = 'UNCERTAIN'
                OR q.L_KIND = 'UNCERTAIN'), 'EXECUTIVE_ROLE_CONTEXT_UNCERTAIN', NULL)
       ) AS LR_REVIEW_FLAGS,
       COALESCE(pp.CANDIDATE_COUNT, 0) AS CANDIDATE_COUNT,
       CASE WHEN pp.CANONICAL_JOB_ID IS NULL THEN 'NO_CANDIDATE'
            WHEN pp.CANDIDATE_COUNT = 1      THEN 'SINGLE_CANDIDATE'
            WHEN pp.N_TOP = 1               THEN 'BEST_OF_MULTIPLE'
            ELSE 'TIED_MULTIPLE' END        AS MATCH_STATUS
FROM LNK_V18_T1_COSMOS t1
LEFT JOIN per_posting pp ON pp.CANONICAL_JOB_ID = t1.CANONICAL_JOB_ID
LEFT JOIN LNK_V18_T2_QC q    ON q.CANONICAL_JOB_ID = t1.CANONICAL_JOB_ID AND q.IS_CHOSEN

## 7 · Automated validation

Additional V18-specific checks verify that selected candidates are assigned the expected rule and tier, and that each rank-1 candidate has the minimum rule number within its posting.

In [ ]:
# every count is wrapped in COALESCE(..., 0) so an empty result reads as 0, never NULL
checks = {
    "Table 3 rows = distinct CANONICAL_JOB_ID in source": """
        SELECT COALESCE((SELECT COUNT(*) FROM LNK_V18_T3_FINAL), 0)
             = COALESCE((SELECT COUNT(DISTINCT CANONICAL_JOB_ID) FROM IDENTIFIER($SRC_COSMOS)), 0)""",
    "Every posting with candidates has exactly one chosen QC row, others none": """
        SELECT COALESCE(COUNT(*), 0) = 0 FROM LNK_V18_T3_FINAL f
        LEFT JOIN (SELECT CANONICAL_JOB_ID, COUNT(*) N FROM LNK_V18_T2_QC WHERE IS_CHOSEN GROUP BY 1) c
               ON c.CANONICAL_JOB_ID = f.CANONICAL_JOB_ID
        WHERE (f.MATCH_STATUS <> 'NO_CANDIDATE' AND COALESCE(c.N, 0) <> 1)
           OR (f.MATCH_STATUS = 'NO_CANDIDATE' AND COALESCE(c.N, 0) <> 0)""",
    "SINGLE_CANDIDATE postings have exactly one candidate": """
        SELECT COALESCE(COUNT(*), 0) = 0 FROM LNK_V18_T3_FINAL
        WHERE MATCH_STATUS = 'SINGLE_CANDIDATE' AND CANDIDATE_COUNT <> 1""",
    "BEST_OF_MULTIPLE postings have 2+ candidates and exactly one ranked first": """
        SELECT COALESCE(COUNT(*), 0) = 0 FROM LNK_V18_T3_FINAL f
        LEFT JOIN (SELECT CANONICAL_JOB_ID, COUNT(*) N FROM LNK_V18_T2_QC WHERE CANDIDATE_RANK = 1 GROUP BY 1) c
               ON c.CANONICAL_JOB_ID = f.CANONICAL_JOB_ID
        WHERE f.MATCH_STATUS = 'BEST_OF_MULTIPLE' AND (f.CANDIDATE_COUNT < 2 OR COALESCE(c.N, 0) <> 1)""",
    "TIED_MULTIPLE postings have 2+ tied top candidates": """
        SELECT COALESCE(COUNT(*), 0) = 0 FROM LNK_V18_T3_FINAL f
        LEFT JOIN (SELECT CANONICAL_JOB_ID, COUNT(*) N FROM LNK_V18_T2_QC WHERE CANDIDATE_RANK = 1 GROUP BY 1) c
               ON c.CANONICAL_JOB_ID = f.CANONICAL_JOB_ID
        WHERE f.MATCH_STATUS = 'TIED_MULTIPLE' AND COALESCE(c.N, 0) < 2""",
    "MATCH_RULE_NO and person empty only for NO_CANDIDATE": """
        SELECT COALESCE(COUNT_IF((MATCH_RULE_NO IS NULL) <> (MATCH_STATUS = 'NO_CANDIDATE')
                              OR (LR_PERSON_ID IS NULL)  <> (MATCH_STATUS = 'NO_CANDIDATE')), 0) = 0
        FROM LNK_V18_T3_FINAL""",
    "No candidate passed with a title conflict": """
        SELECT COALESCE(COUNT_IF(TITLE_CONFLICT), 0) = 0 FROM LNK_V18_T2_QC""",
    "Every word match has the same role level on both sides": """
        SELECT COALESCE(COUNT_IF(TITLE_MATCH_TYPE = 'WORD_MATCH'
                                 AND (MATCHED_POSTING_LEVEL IS NULL OR LEADER_LEVEL IS NULL
                                      OR MATCHED_POSTING_LEVEL <> LEADER_LEVEL)), 0) = 0 FROM LNK_V18_T2_QC""",
    "Tied top candidates share the same rule": """
        SELECT COALESCE(COUNT(*), 0) = 0 FROM (
            SELECT CANONICAL_JOB_ID FROM LNK_V18_T2_QC WHERE CANDIDATE_RANK = 1
            GROUP BY CANONICAL_JOB_ID HAVING COUNT(DISTINCT RULE_NO) > 1)""",
    "MATCH_TIER agrees with match type and similarity": """
        SELECT COALESCE(COUNT_IF(MATCH_TIER <> CASE WHEN TITLE_MATCH_TYPE = 'EXACT'
                                         OR (TITLE_MATCH_TYPE = 'SIMILARITY' AND TITLE_SIM >= $NEAR_EXACT_BAR) THEN 1
                                         WHEN TITLE_MATCH_TYPE = 'SIMILARITY' THEN 2 ELSE 3 END), 0) = 0
        FROM LNK_V18_T2_QC""",
    "Every Table 3 row has a MATCH_STATUS": """
        SELECT COALESCE(COUNT_IF(MATCH_STATUS IS NULL), 0) = 0 FROM LNK_V18_T3_FINAL""",
    "Selected title side passes its own conflict guard": """
        SELECT COALESCE(COUNT_IF(
            (TITLE_MATCHED_ON = 'TITLE_RAW' AND (NOT RAW_QUALIFIES OR RAW_CONFLICT))
            OR (TITLE_MATCHED_ON = 'TITLE_TRANSLATED' AND (NOT TR_QUALIFIES OR TR_CONFLICT))
            OR TITLE_MATCHED_ON IS NULL), 0) = 0
        FROM LNK_V18_T2_QC""",
    "Non-exact similarity does not cross C-suite boundary": """
        SELECT COALESCE(COUNT_IF(TITLE_MATCH_TYPE = 'SIMILARITY'
            AND COALESCE(MATCHED_POSTING_LEVEL = 'C_SUITE', FALSE)
             <> COALESCE(LEADER_LEVEL = 'C_SUITE', FALSE)), 0) = 0
        FROM LNK_V18_T2_QC""",
    "One appointment per posting and person remains": """
        SELECT COALESCE(COUNT(*), 0) = 0 FROM (
            SELECT CANONICAL_JOB_ID, PERSON_ID FROM LNK_V18_T2_QC_BASE
            GROUP BY 1,2 HAVING COUNT(*) > 1)""",
    "No selected title side has an executive-office contradiction": """
        SELECT COALESCE(COUNT_IF(
            (TITLE_MATCHED_ON = 'TITLE_RAW' AND RAW_CONTEXT_CONFLICT)
            OR (TITLE_MATCHED_ON = 'TITLE_TRANSLATED' AND TR_CONTEXT_CONFLICT)
            OR TITLE_CONTEXT_CONFLICT), 0) = 0 FROM LNK_V18_T2_QC""",
    "Every candidate assigned to R1–R8": """
        SELECT COALESCE(COUNT_IF(RULE_NO NOT BETWEEN 1 AND 8 OR RULE_NO IS NULL), 0) = 0
        FROM LNK_V18_T2_QC""",
    "Rule/tier exclusivity (R1,R2,R5,R7 = Tier 1; other rules = Tier 2/3)": """
        SELECT COALESCE(COUNT_IF(
            (RULE_NO IN (1,2,5,7) AND MATCH_TIER <> 1)
            OR (RULE_NO IN (3,4,6,8) AND MATCH_TIER NOT IN (2,3))), 0) = 0
        FROM LNK_V18_T2_QC""",
    "Rule/date windows agree with V18 eligibility": """
        SELECT COALESCE(COUNT_IF(NOT COALESCE(
            (RULE_NO IN (1,3) AND ROLE_START BETWEEN POSTING_MONTH AND DATEADD(month,6,POSTING_MONTH))
            OR (RULE_NO IN (2,4) AND ROLE_START = DATE_FROM_PARTS(YEAR(POSTING_MONTH),1,1))
            OR (RULE_NO IN (5,6) AND ROLE_START BETWEEN POSTING_MONTH AND DATEADD(month,12,POSTING_MONTH))
            OR (RULE_NO IN (7,8) AND ROLE_START < POSTING_MONTH
                AND ROLE_START >= DATEADD(month,-6,POSTING_MONTH)), FALSE)), 0) = 0
        FROM LNK_V18_T2_QC""",
    "Selected candidate has lowest rule for each posting": """
        SELECT COALESCE(COUNT(*),0) = 0 FROM (
            SELECT CANONICAL_JOB_ID
            FROM LNK_V18_T2_QC
            GROUP BY CANONICAL_JOB_ID
            HAVING MIN(IFF(IS_CHOSEN,RULE_NO,NULL)) <> MIN(RULE_NO))""",
    "Tier 1 includes exact or similarity >=98 only": """
        SELECT COALESCE(COUNT_IF(
            (MATCH_TIER = 1) <> (TITLE_MATCH_TYPE = 'EXACT'
               OR (TITLE_MATCH_TYPE = 'SIMILARITY' AND TITLE_SIM >= $NEAR_EXACT_BAR))),0) = 0
        FROM LNK_V18_T2_QC""",
    "Every candidate has a non-null selected title method": """
        SELECT COALESCE(COUNT_IF(TITLE_MATCHED_ON IS NULL), 0) = 0 FROM LNK_V18_T2_QC""",
    "Final match evidence agrees with the chosen QC row": """
        SELECT COALESCE(COUNT_IF(
            f.MATCH_STATUS <> 'NO_CANDIDATE' AND
            (q.PERSON_ID IS NULL OR f.LR_TITLE_MATCH_TYPE IS DISTINCT FROM q.TITLE_MATCH_TYPE
             OR f.LR_TITLE_SIM IS DISTINCT FROM q.TITLE_SIM
             OR f.LR_POST_TO_START_DAYS IS DISTINCT FROM q.POST_TO_START_DAYS)),0) = 0
        FROM LNK_V18_T3_FINAL f
        LEFT JOIN LNK_V18_T2_QC q
          ON q.CANONICAL_JOB_ID = f.CANONICAL_JOB_ID AND q.IS_CHOSEN""",
}
results = {name: session.sql(q).collect()[0][0] is True for name, q in checks.items()}   # NULL counts as a failure
src_after = {SRC_COSMOS: n_rows(SRC_COSMOS), SRC_LEADER: n_rows(SRC_LEADER)}
results["Source row counts unchanged"] = src_after == SRC_COUNTS_BEFORE
for name, ok in results.items():
    print(("PASS  " if ok else "FAIL  ") + name)
assert all(results.values()), "A validation check failed: do not use LNK_V18_T3_FINAL yet"

## 8 · Output diagnostics and review samples

In [ ]:
%%sql -r v18_summary_status
SELECT MATCH_STATUS, COUNT(*) AS POSTINGS, ROUND(100 * RATIO_TO_REPORT(COUNT(*)) OVER (), 1) AS PCT
FROM LNK_V18_T3_FINAL
GROUP BY MATCH_STATUS
ORDER BY POSTINGS DESC

In [ ]:
%%sql -r v18_summary_rules
SELECT MATCH_RULE_NO,
       COALESCE(COUNT_IF(MATCH_STATUS = 'SINGLE_CANDIDATE'), 0) AS SINGLE_CANDIDATE,
       COALESCE(COUNT_IF(MATCH_STATUS = 'BEST_OF_MULTIPLE'), 0) AS BEST_OF_MULTIPLE,
       COALESCE(COUNT_IF(MATCH_STATUS = 'TIED_MULTIPLE'), 0)    AS TIED_MULTIPLE
FROM LNK_V18_T3_FINAL
WHERE MATCH_STATUS <> 'NO_CANDIDATE'
GROUP BY MATCH_RULE_NO
ORDER BY MATCH_RULE_NO

### Rule × tier matrix (selected matches)

This matrix is directly useful for auditing how many predictions come from each rule/tier combination. R1/R2/R5/R7 must be Tier 1; R3/R4/R6/R8 must be Tier 2 or Tier 3.

In [ ]:
%%sql -r v18_rule_by_tier
SELECT q.RULE_NO,
       COUNT_IF(q.MATCH_TIER = 1) AS TIER_1,
       COUNT_IF(q.MATCH_TIER = 2) AS TIER_2,
       COUNT_IF(q.MATCH_TIER = 3) AS TIER_3,
       COUNT(*) AS TOTAL_MATCHES
FROM LNK_V18_T2_QC q
WHERE q.IS_CHOSEN
GROUP BY q.RULE_NO
ORDER BY q.RULE_NO

In [ ]:
%%sql -r v18_summary_title_match_type
-- how the shown person's title matched (exact, 95+ similarity, or word match)
SELECT q.TITLE_MATCH_TYPE, f.MATCH_STATUS, COUNT(*) AS POSTINGS
FROM LNK_V18_T3_FINAL f
JOIN LNK_V18_T2_QC q ON q.CANONICAL_JOB_ID = f.CANONICAL_JOB_ID AND q.IS_CHOSEN
GROUP BY ALL
ORDER BY 1, 2

In [ ]:
%%sql -r v18_summary_match_tier
-- person shown, by match tier (1 exact / near-exact, 2 similarity 95-97, 3 word match)
SELECT q.MATCH_TIER, f.MATCH_STATUS, COUNT(*) AS POSTINGS
FROM LNK_V18_T3_FINAL f
JOIN LNK_V18_T2_QC q ON q.CANONICAL_JOB_ID = f.CANONICAL_JOB_ID AND q.IS_CHOSEN
GROUP BY ALL
ORDER BY 1, 2

In [ ]:
%%sql -r v18_summary_word_match_levels
-- word matches by role level of the person shown
SELECT q.LEADER_LEVEL, COUNT(*) AS POSTINGS
FROM LNK_V18_T3_FINAL f
JOIN LNK_V18_T2_QC q ON q.CANONICAL_JOB_ID = f.CANONICAL_JOB_ID AND q.IS_CHOSEN
WHERE q.TITLE_MATCH_TYPE = 'WORD_MATCH'
GROUP BY 1
ORDER BY 2 DESC

In [ ]:
%%sql -r v18_spot_review
SELECT MATCH_STATUS,
       CASE WHEN MATCH_RULE_NO IN (1,3) THEN 'AFTER_0_6'
            WHEN MATCH_RULE_NO IN (2,4) THEN 'JAN1_FALLBACK'
            WHEN MATCH_RULE_NO IN (5,6) THEN 'AFTER_6_12'
            ELSE 'BEFORE_0_6' END AS RULE_BAND,
       CANONICAL_JOB_ID, COMPANY, TITLE_RAW, POSTING_MONTH,
       LR_PERSON_ID, LR_SOURCE_ROLE_TITLE, LR_ROLE_START, MATCH_RULE_NO, CANDIDATE_COUNT
FROM LNK_V18_T3_FINAL
WHERE MATCH_STATUS IN ('SINGLE_CANDIDATE', 'BEST_OF_MULTIPLE')
QUALIFY ROW_NUMBER() OVER (PARTITION BY MATCH_STATUS, RULE_BAND ORDER BY HASH(CANONICAL_JOB_ID)) <= 50
ORDER BY MATCH_STATUS DESC, RULE_BAND, MATCH_RULE_NO

In [ ]:
%%sql -r v18_spot_review_ties
SELECT q.CANONICAL_JOB_ID, q.TITLE_RAW, q.POSTING_MONTH, q.PERSON_ID, q.SOURCE_ROLE_TITLE, q.ROLE_START,
       q.RULE_NO, q.TITLE_SIM, q.COUNTRY_MATCH, q.DAYS_FROM_POSTING, q.IS_CHOSEN
FROM LNK_V18_T2_QC q
JOIN (SELECT CANONICAL_JOB_ID FROM LNK_V18_T3_FINAL
      WHERE MATCH_STATUS = 'TIED_MULTIPLE'
      QUALIFY ROW_NUMBER() OVER (ORDER BY HASH(CANONICAL_JOB_ID)) <= 25) m
  ON m.CANONICAL_JOB_ID = q.CANONICAL_JOB_ID
WHERE q.CANDIDATE_RANK = 1
ORDER BY q.CANONICAL_JOB_ID, q.IS_CHOSEN DESC, q.PERSON_ID

In [ ]:
%%sql -r v18_cell_34
-- Full-population view: top posting employers by volume and match coverage.
SELECT RCID_NUM, MIN(COMPANY) AS SAMPLE_COMPANY,
       COUNT(*) AS POSTINGS,
       COUNT_IF(MATCH_STATUS <> 'NO_CANDIDATE') AS PREDICTIONS,
       ROUND(100.0 * COUNT_IF(MATCH_STATUS <> 'NO_CANDIDATE') / NULLIF(COUNT(*),0), 1) AS COVERAGE_PCT
FROM LNK_V18_T3_FINAL
GROUP BY RCID_NUM
ORDER BY POSTINGS DESC
LIMIT 100

In [ ]:
%%sql -r dataframe_22
SELECT MATCH_STATUS, COALESCE(q.TITLE_MATCH_TYPE, 'UNMATCHED') AS TITLE_MATCH_TYPE,
       COALESCE(q.MATCH_TIER, 0) AS MATCH_TIER,
       COUNT(*) AS POSTINGS
FROM LNK_V18_T3_FINAL f
LEFT JOIN LNK_V18_T2_QC q
  ON f.CANONICAL_JOB_ID = q.CANONICAL_JOB_ID AND q.IS_CHOSEN
GROUP BY 1,2,3
ORDER BY 1,2,3

In [ ]:
%%sql -r v18_cell_36
SELECT CANONICAL_JOB_ID, COMPANY, RCID_NUM, TITLE_RAW, TITLE_RAW_ROLE,
       POST_DATE, LR_PERSON_ID, LR_SOURCE_ROLE_TITLE, LR_ROLE_START,
       LR_TITLE_MATCH_TYPE, LR_TITLE_SIM, LR_POST_TO_START_DAYS,
       CANDIDATE_COUNT, MATCH_RULE_NO, MATCH_STATUS, LR_REVIEW_FLAGS
FROM LNK_V18_T3_FINAL
ORDER BY CANONICAL_JOB_ID
LIMIT 100

## 9 · Optional V17 → V18 selection comparison

In [ ]:
# OPTIONAL: compare final selections; handles a missing baseline table gracefully.
baseline_table = f"{WORK_SCHEMA}.LNK_V17_T3_FINAL"
current_table  = f"{WORK_SCHEMA}.LNK_V18_T3_FINAL"
try:
    baseline_exists = bool(session.sql(f"SHOW TABLES LIKE 'LNK_V17_T3_FINAL' IN SCHEMA {WORK_SCHEMA}").collect())
except Exception as exc:
    print("Baseline lookup unavailable:", str(exc)[:300]); baseline_exists = False
if baseline_exists:
    comparison = session.sql(f"""
      WITH b AS (SELECT CANONICAL_JOB_ID, LR_PERSON_ID AS OLD_PERSON FROM {baseline_table}),
           n AS (SELECT CANONICAL_JOB_ID, LR_PERSON_ID AS NEW_PERSON FROM {current_table})
      SELECT COUNT(*) AS POSTINGS_IN_NEW,
             COUNT_IF(b.CANONICAL_JOB_ID IS NOT NULL) AS ALSO_IN_BASELINE,
             COUNT_IF(b.OLD_PERSON IS NOT NULL) AS BASELINE_PREDICTIONS,
             COUNT_IF(n.NEW_PERSON IS NOT NULL) AS NEW_PREDICTIONS,
             COUNT_IF(b.OLD_PERSON IS NOT DISTINCT FROM n.NEW_PERSON) AS UNCHANGED_PERSON,
             COUNT_IF(b.OLD_PERSON IS NOT NULL AND n.NEW_PERSON IS NULL) AS LOST_MATCH,
             COUNT_IF(b.OLD_PERSON IS NULL AND n.NEW_PERSON IS NOT NULL) AS NEW_MATCH,
             COUNT_IF(b.OLD_PERSON IS NOT NULL AND n.NEW_PERSON IS NOT NULL
                      AND b.OLD_PERSON IS DISTINCT FROM n.NEW_PERSON) AS REPLACED_PERSON
      FROM n LEFT JOIN b USING (CANONICAL_JOB_ID)
    """).to_pandas()
    print(comparison.to_string(index=False))
else:
    print("Baseline table not found; skipped optional comparison.")

### V17 → V18 changed selections by rule

For every changed prediction, this query shows both leader IDs, their selected rule/tier and status. Run it after the V18 pipeline finishes; it does not verify which selection is correct.

In [ ]:
%%sql -r v18_changed_predictions
SELECT n.CANONICAL_JOB_ID,
       n.COMPANY, n.TITLE_RAW, n.POST_DATE,
       b.LR_PERSON_ID AS V17_PERSON_ID,
       n.LR_PERSON_ID AS V18_PERSON_ID,
       b.MATCH_RULE_NO AS V17_RULE,
       n.MATCH_RULE_NO AS V18_RULE,
       b.LR_MATCH_TIER AS V17_TIER,
       n.LR_MATCH_TIER AS V18_TIER,
       b.MATCH_STATUS AS V17_STATUS,
       n.MATCH_STATUS AS V18_STATUS
FROM LNK_V17_T3_FINAL b
JOIN LNK_V18_T3_FINAL n USING (CANONICAL_JOB_ID)
WHERE b.LR_PERSON_ID IS DISTINCT FROM n.LR_PERSON_ID
ORDER BY n.MATCH_RULE_NO, n.CANONICAL_JOB_ID

In [ ]:
%%sql -r v18_precision_qc_summary
SELECT MATCH_STATUS,
       COALESCE(LR_TITLE_MATCH_TYPE, 'UNMATCHED') AS TITLE_METHOD,
       COUNT(*) AS POSTINGS,
       COUNT_IF(ARRAY_CONTAINS('PRE_POSTING_ROLE_START'::VARIANT, LR_REVIEW_FLAGS)) AS PRE_POSTING_START,
       COUNT_IF(ARRAY_CONTAINS('WORD_MATCH_REVIEW'::VARIANT, LR_REVIEW_FLAGS)) AS WORD_MATCH_FLAGGED,
       COUNT_IF(ARRAY_CONTAINS('TIED_TOP_CANDIDATES'::VARIANT, LR_REVIEW_FLAGS)) AS TIED_FLAGGED
FROM LNK_V18_T3_FINAL
GROUP BY 1, 2
ORDER BY POSTINGS DESC

## Notes and limitations

- **Not executed:** V18 row counts, precision and plausibility cannot be reported until the notebook runs against Snowflake.
- V17 and V18 may select different leaders because rule priority now precedes similarity, country and date proximity.
- R2/R4 Jan-1 fallback is intentionally ranked before some post-advertisement rules, exactly as requested.
- This retains V17's employer gate and executive-office safeguards, but those checks are not a substitute for verified hiring outcomes.
- `LNK_V18_T2_QC` includes all eligible candidate rows and rule/tier evidence; `LNK_V18_T3_FINAL` has one row per canonical posting.

In [ ]:
%%sql -r dataframe_1
select title_raw,title_translated,company,post_date,lr_company_name,lr_source_role_title,match_rule_no from PC_FIVETRAN_DB.DA_WORKSPACE.LNK_V18_T3_FINAL where candidate_count>0